In [ ]:
!pip install dotenv openai pandas openpyxl requests chromadb

In [10]:
from agno.agent import Agent
from agno.models.message import Message
from agno.db.sqlite import SqliteDb
from agno.tools.serper import SerperTools
from agno.models.azure import AzureOpenAI
from agno.vectordb.chroma import ChromaDb
from typing import List
from agno.knowledge.chunking.strategy import ChunkingStrategy
from agno.knowledge.document.base import Document
#from agno.knowledge.content import Document
from agno.knowledge.knowledge import Knowledge
from agno.knowledge.reader.pdf_reader import PDFReader

import os
import pandas as pd
from dotenv import load_dotenv  
from os import getenv
from pathlib import Path  


In [2]:
load_dotenv()
os.environ["AZURE_OPENAI_API_KEY"] = getenv("AZURE_OPENAI_API_KEY")
os.environ["AZURE_ENDPOINT"] = getenv("AZURE_INFERENCE_ENDPOINT")  
os.environ["AZURE_OPENAI_DEPLOYMENT"] = getenv("LLM_MODEL")

In [3]:
model = AzureOpenAI(id=getenv("LLM-MODEL"), 
                    api_version=getenv("LLM_MODEL_VERSION"),
                    azure_endpoint=getenv("AZURE_INFERENCE_ENDPOINT"))
response = model.response(
    messages=[
        Message(role="user", content="Hello")
    ]
)
print(response.content)

Hello! How can I assist you today?


In [4]:
# ─── USER CONFIGURATION ──────────────────────────────────────────────────────
DATA_FOLDER   = Path('../data/input')   # <-- change to your folder

USE_CASE_FILE = DATA_FOLDER / 'use_case_Movie_Recommendation_v2_with_fi_Web_RAG.xlsx'
ABT_FILE      = DATA_FOLDER / 'movie_abt_enriched_FINAL.xlsx'
TAXONOMY_FILE = DATA_FOLDER / 'Movie_Recommendation_TaxonomyV2.xlsx'
OSCARS_PDF    = DATA_FOLDER / 'The_98th_Academy_Awards_2026.pdf'

# SQLite file for Agno session memory
MEMORY_DB     = DATA_FOLDER / 'movie_agent_memory.db'
agent_db = SqliteDb(db_file=MEMORY_DB)


AZURE_OPENAI_API_KEY=os.getenv('AZURE_OPENAI_API_KEY')
AZURE_OPENAI_ENDPOINT=os.getenv('AZURE_OPENAI_ENDPOINT')
AZURE_OPENAI_API_VERSION=os.getenv("LLM_MODEL_VERSION") 
TMDB_API_KEY   = os.getenv('TMDB_API_KEY')
SERPER_API_KEY = os.getenv('SERPER_API_KEY')

LLM_MODEL        = os.getenv("LLM_MODEL")
#EMBED_MODEL      = 'text-embedding-3-small'
MAX_ABT_RESULTS  = 5
MAX_TAG_RESULTS  = 10
RAG_TOP_K        = 4
SESSION_ID       = 'movie-bot-session-001'  # change per user

print(f'Data folder: {DATA_FOLDER.resolve()}')
for f in [USE_CASE_FILE, ABT_FILE, TAXONOMY_FILE, OSCARS_PDF]:
    status = '✅' if f.exists() else '❌ NOT FOUND'
    print(f'  {status}  {f.name}')

Data folder: /Users/nsathi/Library/CloudStorage/Dropbox/Applied AI Institute/Courses/GenerativeAI/G-AI-5-Building-Full-Scale-AI-Solutions/AI-Sandbox/Agno-Class-exercises/data/input
  ✅  use_case_Movie_Recommendation_v2_with_fi_Web_RAG.xlsx
  ✅  movie_abt_enriched_FINAL.xlsx
  ✅  Movie_Recommendation_TaxonomyV2.xlsx
  ✅  The_98th_Academy_Awards_2026.pdf


## Load PDF with PAGE-LEVEL Chunking (Sandbox Method)

**Key Difference**: Instead of splitting by characters, we create 2-page chunks with 1-page overlap.

In [25]:
# Step 1: Read pages as individual documents
reader = PDFReader(chunk=False)  # Disable chunking
page_docs: List[Document] = reader.read(OSCARS_PDF)

print(f"Total pages read: {len(page_docs)}")



DEBUG Reading: The_98th_Academy_Awards_2026

Total pages read: 11


In [32]:
# Step 2: Create sliding window chunks (2 pages, 1-page overlap)
chunked_texts = []
for i in range(len(page_docs) - 1):
    combined_content = page_docs[i].content + "\n" + page_docs[i + 1].content
    chunked_texts.append(combined_content)

print(f"Total chunks created: {len(chunked_texts)}")



Total chunks created: 10


In [ ]:
# Step 3: Insert each chunk as text_content via Knowledge
vector_db = ChromaDb(collection="movie_news", path="tmp/chromadb", persistent_client=True)
knowledge = Knowledge(vector_db=vector_db)

for idx, chunk_text in enumerate(chunked_texts):
    knowledge.insert(
        name=f"pages_{idx+1}_{idx+2}",
        text_content=chunk_text,
        metadata={"pages": f"{idx+1}-{idx+2}"},
    )

DEBUG Embedder not provided, using OpenAIEmbedder as default.

DEBUG Creating Persistent Chroma Client

INFO Adding content from pages_1_2

INFO Selecting reader for extension: Text

DEBUG Reading uploaded file: BytesIO

...

INFO Selecting reader for extension: Text

DEBUG Reading uploaded file: BytesIO

INFO Upserting 1 documents

DEBUG Upserted document: 0666c8bc-b014-42e0-9257-1fc94ad8db8e_1 | pages_10_11 | {'chunk': 1, 'chunk_size': 1551,   
      'pages': '10-11', 'linked_to': '', 'name': 'pages_10_11', 'content_id':                                      
      '2c3232e2-4b0b-5e20-9d0d-887267d22bb6', 'content_hash':                                                      
      '5238212330b475cb493d6d11e9ab12ba95f40cc14b449cb24256071148e994bd'}

DEBUG Committed 1 documents

In [49]:
# Step 4: Use with agent
RAG_agent = Agent(knowledge=knowledge, search_knowledge=True)
response = RAG_agent.run("Which movies were nominated for original screenplay", markdown=True)
print(response.content)

INFO Setting default model to OpenAI Chat

INFO Found 10 documents

The movies nominated for the Original Screenplay category at the 2026 Academy Awards were:

1. **Blue Moon** - Written by Robert Kaplow.
2. **It Was Just an Accident** - Written by Jafar Panahi, with script collaborators Nader Saïvar, Shadmehr Rastin, and Mehdi Mahmoudian.
3. **Marty Supreme** - Written by Ronald Bronstein & Josh Safdie.
4. **Sentimental Value** - Written by Eskil Vogt, Joachim Trier.
5. **Sinners** - Written by Ryan Coogler.


In [ ]:
df_instructions = pd.read_excel(USE_CASE_FILE, sheet_name='Agent Instructions')
AGENT_SYSTEM_PROMPT = df_instructions[df_instructions['Prompt Type'] == 'agent_prompt'].values[0][1]
print('System prompt loaded:')
print(AGENT_SYSTEM_PROMPT)

System prompt loaded:
You are a friendly and knowledgeable movie expert. Your ONLY role is to help users discover and learn about movies. Do not answer questions unrelated to movies.

UNDERSTAND THE USER (Flipped Interaction)
Before recommending any movie, collect the user's needs through a brief, engaging conversation. Ask ONE leading question at a time across these dimensions (do not ask all at once):
  • Viewing context: Are you watching alone, with family, friends, or a date?
  • Audience: Who is the audience — adults, teens, young children, mixed group?
  • Genre mood: What genre or emotional mood are you in? (e.g., action, comedy, thriller, romance, drama, sci-fi, documentary)
  • Decade/era: Any preference for era or decade? (classic, 1980s–90s, modern, recent)
  • Country/language: Any preference for country of origin or spoken language?
  • Oscar/award interest: Are you interested in critically acclaimed or award-nominated films?
Stop asking when you have enough to make a pers

In [52]:
agent = Agent(
    model=model,
    db=agent_db,
    knowledge=knowledge,
    tools=[SerperTools()],
    session_id="My serper chatbot March 3",
    add_history_to_context=True,
    instructions=AGENT_SYSTEM_PROMPT + "\nCurrent memory is: {chat_history}",
    markdown=True,
    debug_mode=False
)

In [53]:
# Normally we use chat bot like a query engine.  Here is a way to test some queries.
test_queries = [
    "Recommend a romantic comedy for date night.",
    "Who are the nominees for Best Picture at the 2026 Oscars?",
    "What are the latest Bollywood movies in 2025?",
    "Is Oppenheimer available on Netflix?",
    "Summarize the chat so far"
    ]

for q in test_queries:
    print(f"\nQuery: {q}")
    response = agent.run(q, stream=False)
    print("\nBot:")
    print(response.content)
    print("-" * 80)


Query: Recommend a romantic comedy for date night.

Bot:
That sounds like a great plan for a date night! To help me recommend the perfect romantic comedy, could you tell me a bit about: 

Are you looking for something more classic or recent?
--------------------------------------------------------------------------------

Query: Who are the nominees for Best Picture at the 2026 Oscars?

Bot:
I found information on the Best Picture nominees for the 2026 Oscars. Here are some of the notable films nominated:

- "Sinners" leading with a record 16 nominations
- "One Battle After Another" with 13 nominations
- "Frankenstein" with 9 nominations
- "Marty Supreme" with 9 nominations
- "Sentimental Value" with 9 nominations
- "Hamnet" with 8 nominations
- "Bugonia" with 4 nominations

"Sinners" seems to be the standout nominee this year with the most nominations overall. If you want, I can give you more details on any of these films or help you find a romantic comedy for your date night! 

Woul

In [ ]:
# This code will demonstrate a real flipped interaction

while True:
    user_input = input("You: ")
    print("Your input: ", user_input)
    
    if user_input.lower().strip() == "exit":
        print("Goodbye!")
        break

    run = agent.run(user_input, stream=False)
    
    print("\nAgent:")
    print(run.content)


    # ✅ Retrieve latest stored memory
    chat_memory = agent.get_chat_history()


    print("\nSession Memory:")
    for m in chat_memory:
        print("-", m.content)


    print("\n" + "-"*50 + "\n")
#Initial query "Can you recommend a movie for a date night.  
# I love romantic movies and my spouse likes historical fiction"

Your input:  I would like to watch movie associated with current wars.  Which is the latest war 

Agent:
I can help you with that! To clarify, by "current wars," are you referring to ongoing or very recent conflicts that have been depicted in movies? Also, are you looking for movies based on a specific recent war, or are you open to films based on any contemporary conflicts? And lastly, do you have a preferred genre or style for these war-related movies (e.g., drama, documentary, action)?

Could you please share a bit more about your preferences?

Session Memory:
- I would like to watch movie associated with current wars.  Which is the latest war 
- I can help you with that! To clarify, by "current wars," are you referring to ongoing or very recent conflicts that have been depicted in movies? Also, are you looking for movies based on a specific recent war, or are you open to films based on any contemporary conflicts? And lastly, do you have a preferred genre or style for these war-rela